In [45]:
# Imports and configuration

import os
import time
import joblib
import warnings

import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import (
    GroupShuffleSplit,
    StratifiedGroupKFold,
    cross_validate,
    ParameterSampler
)

from sklearn.preprocessing import (
    OneHotEncoder,
    MultiLabelBinarizer
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

DATASET_PATH = "../data/processed/nliceCleaned.csv"
MODEL_DIR = "../model"
RESULTS_DIR = "../results"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Configuration loaded.")

Configuration loaded.


In [46]:
# Load cleaned NLICE dataset

cleaned = pd.read_csv(
    DATASET_PATH,
    dtype={
        "PATIENT": "string",
        "AGE": "int16",
        "GENDER": "string",
        "SYMPTOMS_CLEAN": "string",
        "LOCATION": "string",
        "PATHOLOGY": "string"
    }
)

print("Dataset shape:", cleaned.shape)
display(cleaned.head())

Dataset shape: (3692607, 6)


,PATIENT,AGE,GENDER,SYMPTOMS_CLEAN,LOCATION,PATHOLOGY
0,b0915fdc-11f2-430a-aece-4f19f2d03b0f,40,M,Congestion|Discharge|Altered_sensation,Head:Ear|Head:Ear|Head:Ear,Otitis externa
1,bc8b5a00-f5cd-4675-bd26-68ff53101121,58,F,Congestion|Discharge|Altered_sensation,Head:Ear|Head:Ear|Head:Ear,Otitis externa
2,611a810d-eed5-4156-9d98-e375b3754df0,22,M,Inflammation|Congestion|Discharge|Altered_sens...,Head:Ear|Head:Ear|Head:Ear|Head:Ear,Otitis externa
3,41542e6c-0b9c-464e-bb74-00f52b511f07,59,M,Congestion|Discharge|Altered_sensation,Head:Ear|Head:Ear|Head:Ear,Otitis externa
4,280423b0-3df5-416e-b446-0208679936bd,18,F,Congestion|Discharge|Altered_sensation,Head:Ear|Head:Ear|Head:Ear,Otitis externa


In [47]:
# Create locked patient-level 80/20 train-test split

X = cleaned.drop(columns=["PATHOLOGY"])
y = cleaned["PATHOLOGY"]
groups = cleaned["PATIENT"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

train_data = cleaned.iloc[train_idx].copy()
test_data = cleaned.iloc[test_idx].copy()

print("Train shape:", train_data.shape)
print("Test shape :", test_data.shape)

print("\nTrain patients:", train_data["PATIENT"].nunique())
print("Test patients :", test_data["PATIENT"].nunique())

overlap = set(train_data["PATIENT"]) & set(test_data["PATIENT"])

print("Patient overlap:", len(overlap))

assert len(overlap) == 0

Train shape: (2952952, 6)
Test shape : (739655, 6)

Train patients: 440290
Test patients : 110073
Patient overlap: 0


In [48]:
# Verify disease coverage

train_classes = set(train_data["PATHOLOGY"])
test_classes = set(test_data["PATHOLOGY"])

print("Training diseases:", len(train_classes))
print("Testing diseases :", len(test_classes))

missing_in_test = train_classes - test_classes
missing_in_train = test_classes - train_classes

print("Missing from test :", len(missing_in_test))
print("Missing from train:", len(missing_in_train))

assert len(missing_in_test) == 0
assert len(missing_in_train) == 0

print("\nAll disease classes are present in both partitions.")

Training diseases: 55
Testing diseases : 55
Missing from test : 0
Missing from train: 0

All disease classes are present in both partitions.


In [49]:
# Prepare raw features and targets

X_train_raw = train_data.drop(
    columns=["PATIENT", "PATHOLOGY"]
).copy()

X_test_raw = test_data.drop(
    columns=["PATIENT", "PATHOLOGY"]
).copy()

y_train = train_data["PATHOLOGY"].copy()
y_test = test_data["PATHOLOGY"].copy()

groups_train = train_data["PATIENT"].copy()

print("X_train_raw:", X_train_raw.shape)
print("X_test_raw :", X_test_raw.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train_raw: (2952952, 4)
X_test_raw : (739655, 4)
y_train: (2952952,)
y_test : (739655,)


In [50]:
# Encode Age and Gender

age_train = csr_matrix(
    X_train_raw[["AGE"]]
    .astype("float32")
    .values
)

age_test = csr_matrix(
    X_test_raw[["AGE"]]
    .astype("float32")
    .values
)

gender_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

gender_train = gender_encoder.fit_transform(
    X_train_raw[["GENDER"]]
)

gender_test = gender_encoder.transform(
    X_test_raw[["GENDER"]]
)

print("Age features:", age_train.shape[1])
print("Gender features:", gender_train.shape[1])

Age features: 1
Gender features: 2


In [51]:
# Encode Symptoms

X_train_raw["SYMPTOMS_CLEAN"] = (
    X_train_raw["SYMPTOMS_CLEAN"]
    .fillna("")
)

X_test_raw["SYMPTOMS_CLEAN"] = (
    X_test_raw["SYMPTOMS_CLEAN"]
    .fillna("")
)

train_symptom_lists = (
    X_train_raw["SYMPTOMS_CLEAN"]
    .str.split("|")
)

test_symptom_lists = (
    X_test_raw["SYMPTOMS_CLEAN"]
    .str.split("|")
)

symptom_encoder = MultiLabelBinarizer(
    sparse_output=True
)

symptoms_train = symptom_encoder.fit_transform(
    train_symptom_lists
)

symptoms_test = symptom_encoder.transform(
    test_symptom_lists
)

print("Symptom features:", symptoms_train.shape[1])

Symptom features: 80


In [52]:
# Encode Locations

X_train_raw["LOCATION"] = (
    X_train_raw["LOCATION"]
    .fillna("UNKNOWN")
)

X_test_raw["LOCATION"] = (
    X_test_raw["LOCATION"]
    .fillna("UNKNOWN")
)

train_location_lists = (
    X_train_raw["LOCATION"]
    .str.split("|")
)

test_location_lists = (
    X_test_raw["LOCATION"]
    .str.split("|")
)

location_encoder = MultiLabelBinarizer(
    sparse_output=True
)

locations_train = location_encoder.fit_transform(
    train_location_lists
)

locations_test = location_encoder.transform(
    test_location_lists
)

print("Location features:", locations_train.shape[1])

Location features: 137


In [53]:
# Combine all features

X_train = hstack([
    age_train,
    gender_train,
    symptoms_train,
    locations_train
]).tocsr()

X_test = hstack([
    age_test,
    gender_test,
    symptoms_test,
    locations_test
]).tocsr()

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nTotal features:", X_train.shape[1])

X_train: (2952952, 220)
X_test : (739655, 220)

Total features: 220


In [54]:
# Full patient-level grouped and stratified cross-validation

cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("CV strategy:", cv)
print("Number of folds:", cv.n_splits)
print("Grouping: PATIENT")
print("Stratification: PATHOLOGY")

CV strategy: StratifiedGroupKFold(n_splits=3, random_state=42, shuffle=True)
Number of folds: 3
Grouping: PATIENT
Stratification: PATHOLOGY


In [55]:
# Create patient-level tuning subset
#
# This subset comes ONLY from the 80% training partition.
# The locked 20% test set remains completely untouched.

TUNING_PATIENT_FRACTION = 0.20

unique_train_patients = train_data["PATIENT"].unique()

rng = np.random.default_rng(RANDOM_STATE)

tuning_patient_count = int(
    len(unique_train_patients) * TUNING_PATIENT_FRACTION
)

tuning_patients = rng.choice(
    unique_train_patients,
    size=tuning_patient_count,
    replace=False
)

tuning_patient_set = set(tuning_patients)

tuning_mask = train_data["PATIENT"].isin(
    tuning_patient_set
)

tuning_indices = np.flatnonzero(tuning_mask)

X_tune = X_train[tuning_indices]

y_tune = (
    y_train
    .iloc[tuning_indices]
    .reset_index(drop=True)
)

groups_tune = (
    groups_train
    .iloc[tuning_indices]
    .reset_index(drop=True)
)

print("Tuning records :", len(tuning_indices))
print("Tuning patients:", len(tuning_patient_set))
print("Tuning diseases:", y_tune.nunique())

print("\nTuning feature matrix:", X_tune.shape)

Tuning records : 590748
Tuning patients: 88058
Tuning diseases: 55

Tuning feature matrix: (590748, 220)


In [56]:
# Cross-validation used during hyperparameter tuning

tuning_cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("Tuning CV ready.")

Tuning CV ready.


In [57]:
# Evaluation metrics

SCORING = {
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro"
}

print("Metrics:", list(SCORING.keys()))
print("Primary model-selection metric: Macro F1")

Metrics: ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']
Primary model-selection metric: Macro F1


In [58]:
# Decision Tree hyperparameter tuning

dt_parameter_space = {
    "max_depth": [10, 15, 20, 25, 30, None],
    "min_samples_leaf": [1, 2, 5, 10],
    "criterion": ["gini", "entropy"]
}

dt_candidates = list(
    ParameterSampler(
        dt_parameter_space,
        n_iter=8,
        random_state=RANDOM_STATE
    )
)

dt_results = []

for i, params in enumerate(dt_candidates, start=1):

    print(f"\nDecision Tree candidate {i}/{len(dt_candidates)}")
    print(params)

    model = DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        **params
    )

    start = time.time()

    scores = cross_validate(
        model,
        X_tune,
        y_tune,
        groups=groups_tune,
        cv=tuning_cv,
        scoring=SCORING,
        n_jobs=-1,
        return_train_score=False
    )

    elapsed = time.time() - start

    result = {
        "model": "Decision Tree",
        "params": params,
        "accuracy": scores["test_accuracy"].mean(),
        "precision_macro": scores["test_precision_macro"].mean(),
        "recall_macro": scores["test_recall_macro"].mean(),
        "f1_macro": scores["test_f1_macro"].mean(),
        "time_seconds": elapsed
    }

    dt_results.append(result)

    print(
        f"Accuracy: {result['accuracy']:.4f} | "
        f"Macro F1: {result['f1_macro']:.4f}"
    )

dt_results_df = pd.DataFrame(dt_results)

dt_results_df = dt_results_df.sort_values(
    "f1_macro",
    ascending=False
).reset_index(drop=True)

display(dt_results_df)


Decision Tree candidate 1/8
{'min_samples_leaf': 10, 'max_depth': 10, 'criterion': 'entropy'}
Accuracy: 0.8440 | Macro F1: 0.8131

Decision Tree candidate 2/8
{'min_samples_leaf': 1, 'max_depth': 30, 'criterion': 'entropy'}
Accuracy: 0.9501 | Macro F1: 0.9520

Decision Tree candidate 3/8
{'min_samples_leaf': 5, 'max_depth': 10, 'criterion': 'entropy'}
Accuracy: 0.8443 | Macro F1: 0.8134

Decision Tree candidate 4/8
{'min_samples_leaf': 10, 'max_depth': 30, 'criterion': 'entropy'}
Accuracy: 0.9505 | Macro F1: 0.9524

Decision Tree candidate 5/8
{'min_samples_leaf': 1, 'max_depth': 10, 'criterion': 'entropy'}
Accuracy: 0.8443 | Macro F1: 0.8134

Decision Tree candidate 6/8
{'min_samples_leaf': 2, 'max_depth': 25, 'criterion': 'entropy'}
Accuracy: 0.9503 | Macro F1: 0.9522

Decision Tree candidate 7/8
{'min_samples_leaf': 1, 'max_depth': 25, 'criterion': 'gini'}
Accuracy: 0.7408 | Macro F1: 0.6843

Decision Tree candidate 8/8
{'min_samples_leaf': 10, 'max_depth': 30, 'criterion': 'gini'}

,model,params,accuracy,precision_macro,recall_macro,f1_macro,time_seconds
0,Decision Tree,"{'min_samples_leaf': 10, 'max_depth': 30, 'cri...",0.950541,0.952562,0.952430,0.952435,7.408947
1,Decision Tree,"{'min_samples_leaf': 2, 'max_depth': 25, 'crit...",0.950297,0.952424,0.952173,0.952152,6.964484
2,Decision Tree,"{'min_samples_leaf': 1, 'max_depth': 30, 'crit...",0.950051,0.952158,0.951931,0.951982,7.949465
3,Decision Tree,"{'min_samples_leaf': 10, 'max_depth': 30, 'cri...",0.918361,0.893485,0.899808,0.892169,6.338901
4,Decision Tree,"{'min_samples_leaf': 1, 'max_depth': 10, 'crit...",0.844336,0.850843,0.819144,0.813412,5.922356
5,Decision Tree,"{'min_samples_leaf': 5, 'max_depth': 10, 'crit...",0.844282,0.850776,0.819090,0.813356,6.719031
6,Decision Tree,"{'min_samples_leaf': 10, 'max_depth': 10, 'cri...",0.844043,0.850509,0.818856,0.813114,7.534991
7,Decision Tree,"{'min_samples_leaf': 1, 'max_depth': 25, 'crit...",0.740788,0.735627,0.718509,0.684322,7.044257


In [59]:
# Random Forest hyperparameter tuning

rf_parameter_space = {
    "n_estimators": [100, 150, 200],
    "max_depth": [None, 15, 20, 25],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", 0.5]
}

rf_candidates = list(
    ParameterSampler(
        rf_parameter_space,
        n_iter=6,
        random_state=RANDOM_STATE
    )
)

rf_results = []

for i, params in enumerate(rf_candidates, start=1):

    print(f"\nRandom Forest candidate {i}/{len(rf_candidates)}")
    print(params)

    model = RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        **params
    )

    start = time.time()

    scores = cross_validate(
        model,
        X_tune,
        y_tune,
        groups=groups_tune,
        cv=tuning_cv,
        scoring=SCORING,
        n_jobs=1,
        return_train_score=False
    )

    elapsed = time.time() - start

    result = {
        "model": "Random Forest",
        "params": params,
        "accuracy": scores["test_accuracy"].mean(),
        "precision_macro": scores["test_precision_macro"].mean(),
        "recall_macro": scores["test_recall_macro"].mean(),
        "f1_macro": scores["test_f1_macro"].mean(),
        "time_seconds": elapsed
    }

    rf_results.append(result)

    print(
        f"Accuracy: {result['accuracy']:.4f} | "
        f"Macro F1: {result['f1_macro']:.4f}"
    )

rf_results_df = pd.DataFrame(rf_results)

rf_results_df = rf_results_df.sort_values(
    "f1_macro",
    ascending=False
).reset_index(drop=True)

display(rf_results_df)


Random Forest candidate 1/6
{'n_estimators': 150, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': None}
Accuracy: 0.9527 | Macro F1: 0.9544

Random Forest candidate 2/6
{'n_estimators': 200, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'max_depth': 25}
Accuracy: 0.9506 | Macro F1: 0.9520

Random Forest candidate 3/6
{'n_estimators': 100, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 15}
Accuracy: 0.9448 | Macro F1: 0.9401

Random Forest candidate 4/6
{'n_estimators': 100, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None}
Accuracy: 0.9513 | Macro F1: 0.9533

Random Forest candidate 5/6
{'n_estimators': 150, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 15}
Accuracy: 0.7160 | Macro F1: 0.6669

Random Forest candidate 6/6
{'n_estimators': 200, 'min_samples_leaf': 2, 'max_features': 0.5, 'max_depth': 20}
Accuracy: 0.8228 | Macro F1: 0.7802


,model,params,accuracy,precision_macro,recall_macro,f1_macro,time_seconds
0,Random Forest,"{'n_estimators': 150, 'min_samples_leaf': 2, '...",0.952655,0.954577,0.954319,0.954413,50.903516
1,Random Forest,"{'n_estimators': 100, 'min_samples_leaf': 1, '...",0.951338,0.953404,0.953174,0.953268,58.498144
2,Random Forest,"{'n_estimators': 200, 'min_samples_leaf': 5, '...",0.950630,0.952664,0.952189,0.951983,28.938933
3,Random Forest,"{'n_estimators': 100, 'min_samples_leaf': 1, '...",0.944848,0.948562,0.942203,0.940069,15.315937
4,Random Forest,"{'n_estimators': 200, 'min_samples_leaf': 2, '...",0.822769,0.840504,0.797683,0.780217,52.998833
5,Random Forest,"{'n_estimators': 150, 'min_samples_leaf': 1, '...",0.715992,0.729607,0.693257,0.666904,32.333501


In [60]:
# Combine all hyperparameter tuning results

tuning_results = pd.concat(
    [
        dt_results_df,
        rf_results_df
    ],
    ignore_index=True
)

tuning_results = tuning_results.sort_values(
    "f1_macro",
    ascending=False
).reset_index(drop=True)

print("All hyperparameter tuning results:")
display(tuning_results)

All hyperparameter tuning results:


,model,params,accuracy,precision_macro,recall_macro,f1_macro,time_seconds
0,Random Forest,"{'n_estimators': 150, 'min_samples_leaf': 2, '...",0.952655,0.954577,0.954319,0.954413,50.903516
1,Random Forest,"{'n_estimators': 100, 'min_samples_leaf': 1, '...",0.951338,0.953404,0.953174,0.953268,58.498144
2,Decision Tree,"{'min_samples_leaf': 10, 'max_depth': 30, 'cri...",0.950541,0.952562,0.952430,0.952435,7.408947
3,Decision Tree,"{'min_samples_leaf': 2, 'max_depth': 25, 'crit...",0.950297,0.952424,0.952173,0.952152,6.964484
4,Random Forest,"{'n_estimators': 200, 'min_samples_leaf': 5, '...",0.950630,0.952664,0.952189,0.951983,28.938933
5,Decision Tree,"{'min_samples_leaf': 1, 'max_depth': 30, 'crit...",0.950051,0.952158,0.951931,0.951982,7.949465
6,Random Forest,"{'n_estimators': 100, 'min_samples_leaf': 1, '...",0.944848,0.948562,0.942203,0.940069,15.315937
7,Decision Tree,"{'min_samples_leaf': 10, 'max_depth': 30, 'cri...",0.918361,0.893485,0.899808,0.892169,6.338901
8,Decision Tree,"{'min_samples_leaf': 1, 'max_depth': 10, 'crit...",0.844336,0.850843,0.819144,0.813412,5.922356
9,Decision Tree,"{'min_samples_leaf': 5, 'max_depth': 10, 'crit...",0.844282,0.850776,0.819090,0.813356,6.719031


In [61]:
# Select the best hyperparameter configuration for each model family

best_dt = (
    dt_results_df
    .sort_values("f1_macro", ascending=False)
    .iloc[0]
)

best_rf = (
    rf_results_df
    .sort_values("f1_macro", ascending=False)
    .iloc[0]
)

print("Best Decision Tree configuration")
print("---------------------------------")
print("Parameters:", best_dt["params"])
print(f"CV Accuracy : {best_dt['accuracy']:.4f}")
print(f"CV Precision: {best_dt['precision_macro']:.4f}")
print(f"CV Recall   : {best_dt['recall_macro']:.4f}")
print(f"CV Macro F1 : {best_dt['f1_macro']:.4f}")

print("\nBest Random Forest configuration")
print("---------------------------------")
print("Parameters:", best_rf["params"])
print(f"CV Accuracy : {best_rf['accuracy']:.4f}")
print(f"CV Precision: {best_rf['precision_macro']:.4f}")
print(f"CV Recall   : {best_rf['recall_macro']:.4f}")
print(f"CV Macro F1 : {best_rf['f1_macro']:.4f}")

Best Decision Tree configuration
---------------------------------
Parameters: {'min_samples_leaf': 10, 'max_depth': 30, 'criterion': 'entropy'}
CV Accuracy : 0.9505
CV Precision: 0.9526
CV Recall   : 0.9524
CV Macro F1 : 0.9524

Best Random Forest configuration
---------------------------------
Parameters: {'n_estimators': 150, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': None}
CV Accuracy : 0.9527
CV Precision: 0.9546
CV Recall   : 0.9543
CV Macro F1 : 0.9544


In [62]:
# Revalidate the best configuration of BOTH model families
# on the complete 80% training partition.
#
# These are the metrics used for the final DT vs RF comparison.

models_to_revalidate = {
    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        **best_dt["params"]
    ),
    "Random Forest": RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        **best_rf["params"]
    )
}

full_cv_results = []

for model_name, model in models_to_revalidate.items():

    print(f"\nRevalidating {model_name} on full 80% training data...")
    print("Parameters:", model.get_params())

    start = time.time()

    scores = cross_validate(
        model,
        X_train,
        y_train,
        groups=groups_train,
        cv=cv,
        scoring=SCORING,
        n_jobs=1,
        return_train_score=False
    )

    elapsed = time.time() - start

    result = {
        "Model": model_name,
        "Parameters": str(model.get_params()),
        "Accuracy": scores["test_accuracy"].mean(),
        "Macro Precision": scores["test_precision_macro"].mean(),
        "Macro Recall": scores["test_recall_macro"].mean(),
        "Macro F1": scores["test_f1_macro"].mean(),
        "CV Time (seconds)": elapsed
    }

    full_cv_results.append(result)

    print(
        f"Accuracy : {result['Accuracy']:.4f}\n"
        f"Precision: {result['Macro Precision']:.4f}\n"
        f"Recall   : {result['Macro Recall']:.4f}\n"
        f"Macro F1 : {result['Macro F1']:.4f}"
    )

full_cv_results_df = pd.DataFrame(full_cv_results)

full_cv_results_df = full_cv_results_df.sort_values(
    "Macro F1",
    ascending=False
).reset_index(drop=True)

print("\nFull training-set CV comparison:")
display(full_cv_results_df)


Revalidating Decision Tree on full 80% training data...
Parameters: {'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'entropy', 'max_depth': 30, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 10, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'random_state': 42, 'splitter': 'best'}
Accuracy : 0.9528
Precision: 0.9547
Recall   : 0.9546
Macro F1 : 0.9546

Revalidating Random Forest on full 80% training data...
Parameters: {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 2, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 150, 'n_jobs': -1, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}
Accuracy : 0.9532
Precision: 0.9551
Recall   : 0.9549
Macro F1 : 0.9549

Ful

,Model,Parameters,Accuracy,Macro Precision,Macro Recall,Macro F1,CV Time (seconds)
0,Random Forest,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.953189,0.955111,0.954937,0.954940,834.646660
1,Decision Tree,"{'ccp_alpha': 0.0, 'class_weight': None, 'crit...",0.952809,0.954741,0.954589,0.954575,75.364027


In [63]:
# Select final model using full-training grouped CV Macro F1

best_full_cv = full_cv_results_df.iloc[0]

best_model_name = best_full_cv["Model"]

if best_model_name == "Decision Tree":

    best_params = best_dt["params"]

    selected_model = DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        **best_params
    )

elif best_model_name == "Random Forest":

    best_params = best_rf["params"]

    selected_model = RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        **best_params
    )

else:
    raise ValueError(
        f"Unsupported model: {best_model_name}"
    )

print("FINAL MODEL SELECTION")
print("=====================")
print("Model:", best_model_name)
print("Parameters:", best_params)

print(
    f"Full CV Accuracy : "
    f"{best_full_cv['Accuracy']:.4f}"
)

print(
    f"Full CV Precision: "
    f"{best_full_cv['Macro Precision']:.4f}"
)

print(
    f"Full CV Recall   : "
    f"{best_full_cv['Macro Recall']:.4f}"
)

print(
    f"Full CV Macro F1 : "
    f"{best_full_cv['Macro F1']:.4f}"
)

FINAL MODEL SELECTION
Model: Random Forest
Parameters: {'n_estimators': 150, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': None}
Full CV Accuracy : 0.9532
Full CV Precision: 0.9551
Full CV Recall   : 0.9549
Full CV Macro F1 : 0.9549


In [64]:
# Save post-tuning model comparison results

TUNED_COMPARISON_PATH = os.path.join(
    RESULTS_DIR,
    "tuned_model_comparison.csv"
)

full_cv_results_df.to_csv(
    TUNED_COMPARISON_PATH,
    index=False
)

print("Saved:", TUNED_COMPARISON_PATH)
display(full_cv_results_df)

Saved: ../results/tuned_model_comparison.csv


,Model,Parameters,Accuracy,Macro Precision,Macro Recall,Macro F1,CV Time (seconds)
0,Random Forest,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.953189,0.955111,0.954937,0.954940,834.646660
1,Decision Tree,"{'ccp_alpha': 0.0, 'class_weight': None, 'crit...",0.952809,0.954741,0.954589,0.954575,75.364027


In [65]:
# Train final selected model on the complete 80% training data

print("Training final model...")

start = time.time()

final_model = selected_model

final_model.fit(
    X_train,
    y_train
)

final_training_time = time.time() - start

print(
    f"Final training completed in "
    f"{final_training_time:.2f} seconds"
)

Training final model...
Final training completed in 497.87 seconds


In [66]:
# Final evaluation on the locked 20% patient-level test set

print("Evaluating final model...")

start = time.time()

y_pred = final_model.predict(X_test)

prediction_time = time.time() - start

test_accuracy = accuracy_score(
    y_test,
    y_pred
)

test_precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

test_recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

print("\n========== FINAL TEST RESULTS ==========")

print(f"Accuracy        : {test_accuracy:.4f}")
print(f"Macro Precision : {test_precision:.4f}")
print(f"Macro Recall    : {test_recall:.4f}")
print(f"Macro F1        : {test_f1:.4f}")

print(f"\nPrediction time : {prediction_time:.2f} seconds")

Evaluating final model...

========== FINAL TEST RESULTS ==========
Accuracy        : 0.9532
Macro Precision : 0.9551
Macro Recall    : 0.9550
Macro F1        : 0.9550

Prediction time : 8.97 seconds


In [67]:
# Generate final confusion matrix and classification report

class_labels = sorted(
    y_test.unique()
)

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=class_labels
)

cm_df = pd.DataFrame(
    cm,
    index=class_labels,
    columns=class_labels
)

CM_PATH = os.path.join(
    RESULTS_DIR,
    "confusion_matrix.csv"
)

cm_df.to_csv(CM_PATH)

report_dict = classification_report(
    y_test,
    y_pred,
    labels=class_labels,
    output_dict=True,
    zero_division=0
)

classification_report_df = (
    pd.DataFrame(report_dict)
    .transpose()
)

REPORT_PATH = os.path.join(
    RESULTS_DIR,
    "classification_report.csv"
)

classification_report_df.to_csv(
    REPORT_PATH
)

print("Confusion matrix shape:", cm_df.shape)
print("Saved:", CM_PATH)
print("Saved:", REPORT_PATH)

Confusion matrix shape: (55, 55)
Saved: ../results/confusion_matrix.csv
Saved: ../results/classification_report.csv


In [68]:
# Save final model and preprocessing artifacts

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "model.joblib"
)

GENDER_ENCODER_PATH = os.path.join(
    MODEL_DIR,
    "gender_encoder.joblib"
)

SYMPTOM_ENCODER_PATH = os.path.join(
    MODEL_DIR,
    "symptom_encoder.joblib"
)

LOCATION_ENCODER_PATH = os.path.join(
    MODEL_DIR,
    "location_encoder.joblib"
)

joblib.dump(
    final_model,
    MODEL_PATH
)

joblib.dump(
    gender_encoder,
    GENDER_ENCODER_PATH
)

joblib.dump(
    symptom_encoder,
    SYMPTOM_ENCODER_PATH
)

joblib.dump(
    location_encoder,
    LOCATION_ENCODER_PATH
)

print("Saved:")
print(MODEL_PATH)
print(GENDER_ENCODER_PATH)
print(SYMPTOM_ENCODER_PATH)
print(LOCATION_ENCODER_PATH)

Saved:
../model/model.joblib
../model/gender_encoder.joblib
../model/symptom_encoder.joblib
../model/location_encoder.joblib


In [69]:
# Save feature metadata

feature_metadata = {
    "feature_order": [
        "AGE",
        "GENDER",
        "SYMPTOMS_CLEAN",
        "LOCATION"
    ],
    "age_features": 1,
    "gender_categories": (
        gender_encoder.categories_[0].tolist()
    ),
    "symptom_classes": (
        symptom_encoder.classes_.tolist()
    ),
    "location_classes": (
        location_encoder.classes_.tolist()
    ),
    "total_features": X_train.shape[1],
    "random_state": RANDOM_STATE,
    "model_name": best_model_name,
    "model_parameters": best_params,
    "selection_metric": "Macro F1",
    "evaluation_protocol": (
        "Patient-level 80/20 split; "
        "3-fold StratifiedGroupKFold on training data; "
        "locked test set"
    )
}

METADATA_PATH = os.path.join(
    MODEL_DIR,
    "feature_metadata.joblib"
)

joblib.dump(
    feature_metadata,
    METADATA_PATH
)

print("Saved:", METADATA_PATH)

Saved: ../model/feature_metadata.joblib


In [70]:
# Save final test results

final_results = pd.DataFrame([
    {
        "model": best_model_name,
        "accuracy": test_accuracy,
        "precision_macro": test_precision,
        "recall_macro": test_recall,
        "f1_macro": test_f1,
        "training_time_seconds": final_training_time,
        "prediction_time_seconds": prediction_time
    }
])

RESULTS_PATH = os.path.join(
    RESULTS_DIR,
    "model_results.csv"
)

final_results.to_csv(
    RESULTS_PATH,
    index=False
)

display(final_results)

print("\nSaved:", RESULTS_PATH)

,model,accuracy,precision_macro,recall_macro,f1_macro,training_time_seconds,prediction_time_seconds
0,Random Forest,0.953175,0.955114,0.954952,0.955007,497.865464,8.974866



Saved: ../results/model_results.csv


In [71]:
# Verify saved artifacts

print("========== SAVED ARTIFACTS ==========")

for path in [
    MODEL_PATH,
    GENDER_ENCODER_PATH,
    SYMPTOM_ENCODER_PATH,
    LOCATION_ENCODER_PATH,
    METADATA_PATH,
    RESULTS_PATH,
    TUNED_COMPARISON_PATH,
    CM_PATH,
    REPORT_PATH
]:

    exists = os.path.exists(path)

    size_mb = (
        os.path.getsize(path) / (1024 ** 2)
        if exists
        else 0
    )

    print(
        f"{path} | "
        f"exists={exists} | "
        f"size={size_mb:.2f} MB"
    )

========== SAVED ARTIFACTS ==========
../model/model.joblib | exists=True | size=1992.72 MB
../model/gender_encoder.joblib | exists=True | size=0.00 MB
../model/symptom_encoder.joblib | exists=True | size=0.00 MB
../model/location_encoder.joblib | exists=True | size=0.01 MB
../model/feature_metadata.joblib | exists=True | size=0.00 MB
../results/model_results.csv | exists=True | size=0.00 MB
../results/tuned_model_comparison.csv | exists=True | size=0.00 MB
../results/confusion_matrix.csv | exists=True | size=0.01 MB
../results/classification_report.csv | exists=True | size=0.00 MB


In [72]:
# Final inference sanity test and summary

loaded_model = joblib.load(
    MODEL_PATH
)

sample_prediction = loaded_model.predict(
    X_test[:5]
)

print("Sample predictions:")

for i, prediction in enumerate(
    sample_prediction,
    start=1
):
    print(f"{i}. {prediction}")

print("\n==========================================")
print("SYNC CARE - FINAL MODEL PIPELINE")
print("==========================================")

print(f"Dataset records       : {len(cleaned):,}")
print(f"Training records      : {len(train_data):,}")
print(f"Locked test records   : {len(test_data):,}")

print(
    f"\nPatients in training  : "
    f"{train_data['PATIENT'].nunique():,}"
)

print(
    f"Patients in test      : "
    f"{test_data['PATIENT'].nunique():,}"
)

print(f"\nDisease classes       : {y_train.nunique()}")
print(f"Features              : {X_train.shape[1]}")

print(f"\nSelected model        : {best_model_name}")
print(f"Selected parameters   : {best_params}")

print("\nFinal locked-test performance")
print(f"Accuracy              : {test_accuracy:.4f}")
print(f"Macro Precision       : {test_precision:.4f}")
print(f"Macro Recall          : {test_recall:.4f}")
print(f"Macro F1              : {test_f1:.4f}")

print("\nArtifacts saved to:")
print("../model/")
print("../results/")

print("\nML pipeline complete.")

Sample predictions:
1. Otitis externa
2. Otitis externa
3. Otitis externa
4. Otitis externa
5. Otitis externa

SYNC CARE - FINAL MODEL PIPELINE
Dataset records       : 3,692,607
Training records      : 2,952,952
Locked test records   : 739,655

Patients in training  : 440,290
Patients in test      : 110,073

Disease classes       : 55
Features              : 220

Selected model        : Random Forest
Selected parameters   : {'n_estimators': 150, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': None}

Final locked-test performance
Accuracy              : 0.9532
Macro Precision       : 0.9551
Macro Recall          : 0.9550
Macro F1              : 0.9550

Artifacts saved to:
../model/
../results/

ML pipeline complete.
